# Non-Spelling SAE Absorption — Selection-Isolation & AUC-Difference CIs (Demo)

A **minimal, runnable demo** of the iter-3 re-analysis from
*"Non-Spelling SAE Absorption: RE-k Selection-Isolation, AUC-Diff CIs, Router Inputs."*

**Problem.** Single SAE latents are unreliable units of analysis — *feature absorption*,
*feature splitting*, non-atomicity — so simple baselines often beat raw-latent SAE methods.
The method builds a small, human-auditable **K-track unit** (a high-recall *anchor* latent plus
a few mutually-exclusive *absorber* latents recovered by greedy set-cover) and asks whether that
**cluster-level unit** is a genuinely better detector of a content attribute than:

* (i) raw single latents (the `anchor`),
* (ii) attribution-ranked latent pools `(g)` (top-20) and `(h)` (count-matched to the unit),
* (iii) a **non-SAE** dense logistic probe on the raw residual stream (`dense_probe`), and
* (iv) **random eligible-latent pools of the same size** — the `RE-k` / `RE-k-anchored` controls
  that separate genuine *selection* of absorbers from mere eligibility + pooling.

**What this demo computes** (verbatim re-use of the artifact's analysis functions on a curated
subset of cached **Gemma-Scope `layer_12/width_16k`** SAE detector scores):

| | analysis |
|---|---|
| **M2** | per-slice classification **AUC** for every detector + **paired-bootstrap AUC-difference CIs** |
| **M1** | the **RE-k / RE-k-anchored** selection-isolation draws + the `selection_established` rule |
| **R1** | a comparison-matched **Youden** accuracy table (no detector forced to predict-all) |
| **router** | the firing-Jaccard **absorption-vs-splitting** regime label |

**Headline (taxonomic hierarchy).** The country **Georgia** (the homograph US-state / country) is
the *defining absorbed slice*: the unit AUC ≈ 0.99 while the attribution pools `(g)/(h)` fall
**below chance** (the absorption signature), and the unit beats the RE-k-anchored random control —
so `selection_established(Georgia) = True`. **United States** (co-firing, not absorbed) and
**Jordan** (descriptive, n<150) are honest contrasts.

> Runs entirely on **pre-computed detector scores** in `mini_demo_data.json` — no GPU, no gated
> model, no SAE re-encode. The heavy upstream steps (loading Gemma-2-2b, SAE encoding of the
> residual stream) produced these scores and are *not* part of this re-analysis demo.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Every package below is pre-installed on Colab. On Colab: skip (installing would corrupt the
# pre-loaded C extensions). Locally: install at Colab's exact versions to mirror the environment.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

In [ ]:
import json, os
import numpy as np
from scipy.stats import rankdata
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

# numpy 2.0 compatibility shims (harmless if the attrs already exist)
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "product"): np.product = np.prod

SEED = 20240617                       # matches the artifact's method.SEED
rng = np.random.default_rng(SEED)     # module-global RNG used by the verbatim functions
print("numpy", np.__version__)

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ee30c-catching-silent-feature-absorption-in-fr/main/round-3/experiment-2/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
meta = data["metadata"]
rows = data["rows"]
print(f"loaded {len(rows)} rows | hierarchy={meta['hierarchy']} | defining slice={meta['defining_slice']}")
print(f"unit latents = {meta['unit_latents']} (anchor={meta['anchor_latent']}), k={meta['k']}")
print(f"|ELIG| eligible-cover pool = {len(meta['elig_latents'])} latents ({meta['elig_relaxation']})")

## Configuration

All tunable knobs live here. They start near the **minimum** that produces meaningful output and
can be scaled up toward the artifact's published settings (shown in comments). On this curated
subset even the full settings finish in a few seconds.

In [ ]:
# --- bootstrap / draw counts ---
# The curated subset is small, so the artifact's FULL settings finish in a few seconds.
# For an instant smoke run, drop to B_AUC=200 / B_DRAWS=50 (the RE-k 95th-pct null is then noisy,
# which can spuriously flip a borderline slice such as United States).
B_AUC   = 10000    # paired-bootstrap reps for AUC-difference CIs   (artifact full setting)
B_DRAWS = 1000     # RE-k random eligible-pool draws                (artifact full setting)

# --- which sub-context slices to analyse (all within the ONE taxonomic dataset) ---
SLICES = ["Georgia", "United States", "Jordan"]   # win / co-firing / descriptive contrasts

# thresholds (pinned from the artifact dossiers; carried in the data file)
JACCARD_MAX    = meta["thresholds"]["jaccard_max"]
N_MIN_ELIGIBLE = meta["thresholds"]["n_min_eligible"]
print(f"B_AUC={B_AUC}  B_DRAWS={B_DRAWS}  slices={SLICES}  JACCARD_MAX={JACCARD_MAX}")

## Unpack the curated subset into detector-score arrays

Each row carries the pre-computed continuous score of every detector plus the sparse firing of
the **eligible-cover (ELIG)** latent pool. We rebuild the dense `[n_rows, |ELIG|]` activation
matrix — it is what the RE-k random draws sample from. As a sanity check, the shipped `unit`
score is exactly the pooled-max over the unit latents' ELIG columns (`pool_score`).

In [ ]:
# per-row label (1 = attribute present) and sub-context
y      = np.array([r["label"] for r in rows], dtype=int)
subs   = np.array([r["sub"] for r in rows], dtype=object)
n_rows = len(rows)

# pre-computed continuous detector scores (length n_rows) for the 5 base detectors:
#   unit        = pooled-max over the K-track unit latents (anchor + recovered absorbers)
#   anchor      = the single high-recall parent latent
#   g           = pooled-max over the top-20 attribution latents (SCR/TPP oracle pool)
#   h           = pooled-max over the top-k attribution latents (count-matched to the unit)
#   dense_probe = a NON-SAE logistic probe on the raw residual stream
det_scores = {dk: np.array([r["scores"][dk] for r in rows], dtype=float)
              for dk in meta["detectors"]}

# sparse ELIG firing -> dense [n_rows, |ELIG|] activation matrix (drives the RE-k draws)
n_elig = len(meta["elig_latents"])
elig_mat = np.zeros((n_rows, n_elig), dtype=np.float32)
for i, r in enumerate(rows):
    for col, val in r["elig_fire"]:
        elig_mat[i, col] = val
anchor_col = meta["anchor_elig_col"]
unit_cols  = meta["unit_elig_cols"]

# sanity: the 'unit' detector IS the pooled-max over the unit latents' ELIG columns
recon_unit = elig_mat[:, unit_cols].max(1)
print("unit == pooled-max over ELIG columns:",
      np.allclose(recon_unit, det_scores["unit"], atol=1e-4))
print("detector arrays:", {k: v.shape for k, v in det_scores.items()},
      "| elig_mat:", elig_mat.shape)

## Analysis functions — copied verbatim from the artifact's `method.py`

`fast_auc`, `_auc_rows` (vectorised bootstrap AUC via rank-sum), `_youden_table`
(comparison-matched operating point), and `firing_jaccard_pos` are reproduced **unchanged**.

In [ ]:
def fast_auc(pos_scores: np.ndarray, neg_scores: np.ndarray) -> float:
    """AUC = P(pos>neg) + 0.5*P(tie) via searchsorted (ties handled). NaN if a class empty."""
    n_pos = len(pos_scores)
    n_neg = len(neg_scores)
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ns = np.sort(neg_scores)
    left = np.searchsorted(ns, pos_scores, side="left")    # count neg < pos
    right = np.searchsorted(ns, pos_scores, side="right")  # count neg <= pos
    less = float(left.sum())
    equal = float((right - left).sum())
    return (less + 0.5 * equal) / (n_pos * n_neg)


def _auc_rows(pos_mat: np.ndarray, neg_mat: np.ndarray) -> np.ndarray:
    """Vectorised AUC per bootstrap row via average-tie rank-sum (Mann-Whitney).
    pos_mat [B,n_pos], neg_mat [B,n_neg] -> AUC [B]. Constant rows -> 0.5 (all ties)."""
    from scipy.stats import rankdata
    B, n_pos = pos_mat.shape
    n_neg = neg_mat.shape[1]
    comb = np.concatenate([pos_mat, neg_mat], axis=1)
    ranks = rankdata(comb, axis=1)            # average ties, per row
    sum_pos = ranks[:, :n_pos].sum(1)
    return (sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


def _youden_table(det_scores: dict, pos_idx: np.ndarray, neg_idx: np.ndarray) -> dict:
    """Per-detector Youden-J operating point: fit tau on a 50/50 split, eval on the held-out
    half. Reports accuracy / TPR / FPR / balanced-acc + a predicts_all flag so NO detector is
    forced to predict-all (the failure mode of the matched-recall table)."""
    from sklearn.metrics import roc_curve
    pos = pos_idx.copy(); neg = neg_idx.copy()
    rng.shuffle(pos); rng.shuffle(neg)
    ph, nh = len(pos) // 2, len(neg) // 2
    fit_pos, ev_pos = pos[:ph], pos[ph:]
    fit_neg, ev_neg = neg[:nh], neg[nh:]
    if min(len(fit_pos), len(ev_pos), len(fit_neg), len(ev_neg)) == 0:
        return {"note": "insufficient_for_split", "n_pos": int(len(pos)), "n_neg": int(len(neg))}
    out = {"n_pos": int(len(pos)), "n_neg": int(len(neg)), "detectors": {}}
    yfit = np.r_[np.ones(len(fit_pos)), np.zeros(len(fit_neg))]
    for det, sc in det_scores.items():
        sfit = np.r_[sc[fit_pos], sc[fit_neg]]
        fpr, tpr, thr = roc_curve(yfit, sfit)
        tau = float(thr[int(np.argmax(tpr - fpr))])
        ev_tpr = float((sc[ev_pos] >= tau).mean())
        ev_fpr = float((sc[ev_neg] >= tau).mean())
        acc = float((np.r_[sc[ev_pos] >= tau, sc[ev_neg] < tau]).mean())
        out["detectors"][det] = {
            "tau": tau, "accuracy": acc, "tpr": ev_tpr, "fpr": ev_fpr,
            "balanced_acc": float((ev_tpr + (1.0 - ev_fpr)) / 2.0),
            "predicts_all_positive": bool(ev_fpr >= 0.99 and ev_tpr >= 0.99),
        }
    return out


def firing_jaccard_pos(fire_a: np.ndarray, fire_b: np.ndarray) -> float:
    """Positive-only firing Jaccard over a boolean firing vector (matches the M4 router def)."""
    inter = int((fire_a & fire_b).sum())
    union = int((fire_a | fire_b).sum())
    return inter / union if union > 0 else 0.0

## M1 (PHASE D) — build the RE-k / RE-k-anchored selection-isolation pools

* **RE-k** — a random size-`k` pool drawn from *all* eligible latents.
* **RE-k-anchored** — holds the high-recall **anchor** fixed + `(k-1)` random eligible absorbers.

Comparing the two-track unit against these isolates the value of *which* absorbers the set-cover
selected, over and above eligibility + pooling. (Increase `B_DRAWS` for a tighter null.)

In [ ]:
k = meta["k"]
elig_all     = np.arange(n_elig)
elig_no_anch = np.array([c for c in range(n_elig) if c != anchor_col])

rek_scores      = np.zeros((B_DRAWS, n_rows), dtype=np.float32)
rek_anch_scores = np.zeros((B_DRAWS, n_rows), dtype=np.float32)
for d in range(B_DRAWS):
    cols = rng.choice(elig_all, size=k, replace=False)
    rek_scores[d] = elig_mat[:, cols].max(1)
    cols_a = [anchor_col] + rng.choice(elig_no_anch, size=k - 1, replace=False).tolist()
    rek_anch_scores[d] = elig_mat[:, cols_a].max(1)

# the per-row mean RE-k / RE-k-anchored scores become two extra "detectors"
det_scores["rek_mean"]      = rek_scores.mean(0)
det_scores["rek_anch_mean"] = rek_anch_scores.mean(0)
print(f"built {B_DRAWS} RE-k and RE-k-anchored draws (k={k})")

## M2 + M1 — per-slice AUC, AUC-difference CIs, RE-k distribution, `selection_established`

For each slice we compute (1) the point AUC of every detector, (2) the stratified paired-bootstrap
CI on `AUC(unit) - AUC(comparison)` (resampling positives and negatives separately), and (3) the
RE-k / RE-k-anchored draw-AUC distribution. The verdict rule (verbatim):

> `selection_established(s) = (unit AUC > RE-k-anchored 95th pct) AND (CI of unit−RE-k-anchored-mean excludes 0)`

In [ ]:
neg_idx = np.where(y == 0)[0]                       # shared negative pool
neg_of  = {det: det_scores[det][neg_idx] for det in det_scores}
comparisons = ["g", "h", "rek_mean", "rek_anch_mean", "dense_probe"]
eligible_subs = set(meta["eligible_subcontexts"])

auc_point, auc_diff, rek_dist, selection_established = {}, {}, {}, {}
for s in SLICES:
    pos_idx = np.where((y == 1) & (subs == s))[0]
    n_pos = len(pos_idx)
    if n_pos == 0:
        print(f"  [skip] {s}: no positives in subset"); continue
    elig_flag = (s in eligible_subs) and (n_pos >= N_MIN_ELIGIBLE)

    # ---- M2: point AUC for every detector ----
    ap = {det: fast_auc(det_scores[det][pos_idx], neg_of[det]) for det in det_scores}
    auc_point[s] = {"n_pos": n_pos, "n_neg": len(neg_idx), "eligible": elig_flag, **ap}

    # ---- M2: stratified paired-bootstrap AUC-DIFFERENCE CIs (unit - comparison) ----
    ip   = rng.integers(0, n_pos,        size=(B_AUC, n_pos))
    ineg = rng.integers(0, len(neg_idx), size=(B_AUC, len(neg_idx)))
    da = _auc_rows(det_scores["unit"][pos_idx][ip], neg_of["unit"][ineg])
    diffs = {}
    for comp in comparisons:
        db = _auc_rows(det_scores[comp][pos_idx][ip], neg_of[comp][ineg])
        d = da - db
        lo, hi = np.percentile(d, [2.5, 97.5])
        diffs[comp] = {"diff": float(ap["unit"] - ap[comp]), "ci_lo": float(lo), "ci_hi": float(hi)}
    auc_diff[s] = diffs

    # ---- M1: RE-k draw AUC distribution + selection rule ----
    rd = {}
    for variant, mat in (("rek", rek_scores), ("rek_anchored", rek_anch_scores)):
        draw_aucs = _auc_rows(mat[:, pos_idx], mat[:, neg_idx])
        draw_aucs = draw_aucs[np.isfinite(draw_aucs)]
        p95 = float(np.percentile(draw_aucs, 95))
        rd[variant] = {"mean": float(draw_aucs.mean()), "p95": p95,
                       "unit_auc": float(ap["unit"]),
                       "unit_pctile": float((draw_aucs < ap["unit"]).mean() * 100),
                       "unit_above_95th": bool(ap["unit"] > p95)}
    rek_dist[s] = rd
    selection_established[s] = bool(rd["rek_anchored"]["unit_above_95th"]
                                    and diffs["rek_anch_mean"]["ci_lo"] > 0)

print("selection_established:", selection_established)

## R1 — comparison-matched Youden accuracy table (defining slice)

A Youden-J operating point is fit on a 50/50 split and evaluated on the held-out half, so **no
detector is forced to predict-all** (the artefact that made `(h)` look degenerate in iter-2).

In [ ]:
defining = meta["defining_slice"]
def_pos_idx = np.where((y == 1) & (subs == defining))[0]
youden = _youden_table(det_scores, def_pos_idx, neg_idx)
print(f"Youden — defining slice '{defining}' "
      f"(n_pos={youden['n_pos']}, n_neg={youden['n_neg']})")

## Router — firing-Jaccard absorption-vs-splitting regime

A slice is **absorption (mutually exclusive)** when the top specialist latent rarely co-fires
with the parent (firing-Jaccard `< JACCARD_MAX`) while the parent has a real recall hole there;
otherwise it is **co-firing (splitting)**. This W_dec-free signal feeds the M4 router table.

In [ ]:
parent_fire = elig_mat[:, anchor_col] > 0
cand_cols   = np.array([c for c in range(n_elig) if c != anchor_col])   # detectors exclude the anchor
router = {}
for s in SLICES:
    msk = (y == 1) & (subs == s)
    if msk.sum() == 0: continue
    rec = (elig_mat[msk][:, cand_cols] > 0).mean(0)        # per-eligible-latent recall on slice s
    top_col  = int(cand_cols[int(np.argmax(rec))])
    det_fire = elig_mat[:, top_col] > 0
    fj = firing_jaccard_pos(parent_fire, det_fire)
    parent_recall = float(parent_fire[msk].mean())
    router[s] = {"firing_jaccard": round(fj, 3),
                 "regime": "absorption(mutually_exclusive)" if fj < JACCARD_MAX else "co_firing(splitting)",
                 "parent_recall_hole": round(1.0 - parent_recall, 3),
                 "n_pos": int(msk.sum())}
for s, e in router.items():
    print(f"  {s:16s} {e}")

## Results — demo vs. published, and the selection-isolation picture

We print the per-slice AUC table, the Georgia AUC-difference CIs, the Youden table, and the
`selection_established` verdict (demo vs. the artifact's published values, which were computed
on the full corpus with `B_AUC=10000`, `B_DRAWS=1000`). Then two figures: a grouped AUC bar
chart and the RE-k-anchored draw distribution for Georgia with the unit AUC marked.

In [ ]:
ref = meta["paper_reference"]

# ---- per-slice AUC table (demo vs published) ----
print("="*92)
print("PER-SLICE AUC  (demo  |  published)        unit / anchor / (g) / (h) / dense / RE-k / RE-k-anch")
print("="*92)
order = ["unit", "anchor", "g", "h", "dense_probe", "rek_mean", "rek_anch_mean"]
for s in SLICES:
    if s not in auc_point: continue
    ap = auc_point[s]
    pub = (ref.get(s) or {}).get("auc_point") or {}
    demo_str = " ".join(f"{ap[d]:.2f}" for d in order)
    pub_str  = " ".join(f"{pub.get('auc_'+d, float('nan')):.2f}" for d in order)
    print(f"{s:14s} n_pos={ap['n_pos']:>3} elig={str(ap['eligible']):>5}")
    print(f"   demo : {demo_str}")
    print(f"   paper: {pub_str}")

# ---- Georgia AUC-difference CIs ----
print("\n" + "="*70)
print(f"AUC-DIFFERENCE CIs  unit - X   on defining slice '{defining}'")
print("="*70)
for comp in comparisons:
    c = auc_diff[defining][comp]
    excl = "excludes 0" if (c["ci_lo"] > 0 or c["ci_hi"] < 0) else "includes 0"
    print(f"  unit - {comp:14s} = {c['diff']:+.3f}  CI[{c['ci_lo']:+.3f}, {c['ci_hi']:+.3f}]  ({excl})")

# ---- Youden table ----
print("\n" + "="*70)
print(f"YOUDEN comparison-matched accuracy  (slice '{defining}')")
print("="*70)
print(f"  {'detector':14s} {'acc':>6} {'bal_acc':>8} {'tpr':>6} {'fpr':>6} {'predicts_all':>13}")
for det, e in youden.get("detectors", {}).items():
    print(f"  {det:14s} {e['accuracy']:>6.2f} {e['balanced_acc']:>8.2f} "
          f"{e['tpr']:>6.2f} {e['fpr']:>6.2f} {str(e['predicts_all_positive']):>13}")

# ---- verdict ----
print("\n" + "="*70)
print("SELECTION-ESTABLISHED VERDICT  (demo vs published)")
print("="*70)
for s in SLICES:
    pub_se = (ref.get(s) or {}).get("selection_established")
    print(f"  {s:16s} demo={str(selection_established.get(s)):>5}   published={pub_se}")
verdict = ("taxonomic_selection_established" if selection_established.get(defining)
           else "eligibility_pooling_only")
print(f"\nVERDICT: {verdict}")

In [ ]:
# ---- figures ----
fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5))

# (left) grouped AUC bar chart: detectors x slices
dets = ["unit", "anchor", "g", "h", "dense_probe", "rek_anch_mean"]
xs = np.arange(len(dets)); w = 0.8 / len(SLICES)
colors = ["#2c7fb8", "#d95f02", "#1b9e77"]
for i, s in enumerate(SLICES):
    if s not in auc_point: continue
    vals = [auc_point[s][d] for d in dets]
    axL.bar(xs + i*w, vals, w, label=s, color=colors[i % len(colors)])
axL.axhline(0.5, ls="--", c="grey", lw=1, label="chance")
axL.set_xticks(xs + w*(len(SLICES)-1)/2)
axL.set_xticklabels([d.replace("_", "\n") for d in dets], fontsize=9)
axL.set_ylabel("AUC"); axL.set_ylim(0, 1.05)
axL.set_title("Detector AUC by slice\n(g)/(h) below chance on Georgia = absorption signature")
axL.legend(fontsize=9)

# (right) RE-k-anchored draw distribution for the defining slice + unit AUC marker
gpos = np.where((y == 1) & (subs == defining))[0]
draw_aucs = _auc_rows(rek_anch_scores[:, gpos], rek_anch_scores[:, neg_idx])
draw_aucs = draw_aucs[np.isfinite(draw_aucs)]
unit_auc = auc_point[defining]["unit"]
p95 = np.percentile(draw_aucs, 95)
axR.hist(draw_aucs, bins=20, color="#9ecae1", edgecolor="white", label="RE-k-anchored draws")
axR.axvline(p95, ls="--", c="black", lw=1.5, label=f"draws 95th pct = {p95:.3f}")
axR.axvline(unit_auc, c="#d95f02", lw=2.5, label=f"K-track unit AUC = {unit_auc:.3f}")
axR.set_xlabel("AUC"); axR.set_ylabel("draw count")
axR.set_title(f"Selection-isolation on '{defining}'\nunit beats random eligible-pool control")
axR.legend(fontsize=9)

plt.tight_layout()
plt.savefig("demo_results.png", dpi=110, bbox_inches="tight")
plt.show()
print("saved demo_results.png")